In [3]:

from pathlib import Path
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

BASE = Path(r"C:\PHJ_project_20261019\DeepLearning_Project_1102")
DATA = BASE / "dataset" / "final_split"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

print("TensorFlow:", tf.__version__)
print("Train:", (DATA / "train").exists())
print("Validation:", (DATA / "val").exists())
print("Test:", (DATA / "test").exists())

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds.class_names
print("Class names:", class_names)

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

# 사전학습 CNN 불러오기
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# 기존 CNN 가중치 고정
base_model.trainable = False

# 모델 구성
inputs = layers.Input(shape=(224, 224, 3))

# MobileNetV2 입력에 맞게 픽셀값 변환
x = layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1
)(inputs)

# CNN 특징 추출
x = base_model(x, training=False)

# 분류층 추가
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()



TensorFlow: 2.21.0
Train: True
Validation: True
Test: True
Found 8928 files belonging to 2 classes.
Found 1115 files belonging to 2 classes.
Found 1116 files belonging to 2 classes.
Class names: ['Disengagement', 'Engagement']


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:

# 학습 중 검증 성능이 개선되지 않으면 조기 종료
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop]
)

# 모델 저장
MODEL_PATH = BASE / "models" / "engagement_mobilenetv2.keras"

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

model.save(MODEL_PATH)

print("학습 모델 저장 완료:", MODEL_PATH)


Epoch 1/5


c:\PHJ_project_20261019\.venv-1\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


558/558 ━━━━━━━━━━━━━━━━━━━━ 178s 309ms/step - accuracy: 0.7538 - loss: 0.4970 - val_accuracy: 0.6852 - val_loss: 0.5789
Epoch 2/5
558/558 ━━━━━━━━━━━━━━━━━━━━ 161s 289ms/step - accuracy: 0.8691 - loss: 0.3350 - val_accuracy: 0.6906 - val_loss: 0.5821
Epoch 3/5
558/558 ━━━━━━━━━━━━━━━━━━━━ 162s 290ms/step - accuracy: 0.8952 - loss: 0.2758 - val_accuracy: 0.7202 - val_loss: 0.5660
Epoch 4/5
558/558 ━━━━━━━━━━━━━━━━━━━━ 163s 293ms/step - accuracy: 0.8987 - loss: 0.2585 - val_accuracy: 0.7309 - val_loss: 0.5550
Epoch 5/5
558/558 ━━━━━━━━━━━━━━━━━━━━ 163s 291ms/step - accuracy: 0.9168 - loss: 0.2288 - val_accuracy: 0.7291 - val_loss: 0.5572
학습 모델 저장 완료: C:\PHJ_project_20261019\DeepLearning_Project_1102\models\engagement_mobilenetv2.keras


In [ ]:

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

# 사전학습 CNN 불러오기
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# 기존 CNN 가중치 고정
base_model.trainable = False

# 모델 구성
inputs = layers.Input(shape=(224, 224, 3))

# MobileNetV2 입력에 맞게 픽셀값 변환
x = layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1
)(inputs)

# CNN 특징 추출
x = base_model(x, training=False)

# 분류층 추가
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()
